In [31]:
import pandas as pd
import numpy as np

In [ ]:
# for google colab
# from google.colab import drive
#drive.mount('/content/drive')

In [32]:
# for google colab
#data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/data/task_2_data_ex - task_2_data_ex.csv.csv", delimiter=",")
data = pd.read_csv("data/task_2_data_ex - task_2_data_ex.csv.csv", delimiter=",")
data.head()

,year,month,produced_material,produced_material_production_type,produced_material_release_type,produced_material_quantity,component_material,component_material_production_type,component_material_release_type,component_material_quantity,plant_id
0,2024,1,10000,8002,FIN,990.00,50000,8002.0,PROD,990.00,RLT_10
1,2024,1,50000,8002,PROD,859.00,80070,8007.0,PROD,879.00,RLT_10
2,2024,1,50000,8002,PROD,859.00,90000,NaN,ADD,50.00,RLT_10
3,2024,1,50000,8002,PROD,859.00,90001,NaN,ADD,20.00,RLT_10
4,2024,1,80070,8007,PROD,929.00,80010,8001.0,PROD,"3,626.00",RLT_10


**I assume each material has a consistent process every month. If the process changes, it must be assigned a different MaterialID.**
Therefore, I have grouped all stages of the process and calculated the total quantity for the entire year to eliminate the month field.

In [33]:
ds = data.copy()
ds['produced_material_quantity'] = pd.to_numeric(ds['produced_material_quantity'].str.replace(',', ''), errors='coerce')
ds['component_material_quantity'] = pd.to_numeric(ds['component_material_quantity'].str.replace(',', ''), errors='coerce')
ds

data_grouped = ds.groupby([
    'year',
    'produced_material',
    'produced_material_production_type',
    'produced_material_release_type',
    'component_material',
    'component_material_production_type',
    'component_material_release_type',
    'plant_id'
], dropna=False).agg({
    'produced_material_quantity': 'sum',
    'component_material_quantity': 'sum'
}).reset_index()

data_grouped.head(15)

,year,produced_material,produced_material_production_type,produced_material_release_type,component_material,component_material_production_type,component_material_release_type,plant_id,produced_material_quantity,component_material_quantity
0,2024,10000,8002,FIN,50000,8002.0,PROD,RLT_10,11708.0,11708.0
1,2024,10001,8002,FIN,50001,8002.0,PROD,RLT_10,12023.0,12023.0
2,2024,10002,8002,FIN,50002,8002.0,PROD,RLT_14,12067.0,12067.0
3,2024,10003,8002,FIN,50003,8002.0,PROD,RLT_14,12091.0,12091.0
4,2024,10004,8002,FIN,50004,8002.0,PROD,RLT_14,12091.0,12091.0
5,2024,10005,8002,FIN,50005,8002.0,PROD,RLT_14,12023.0,12023.0
6,2024,10006,8002,FIN,50006,8002.0,PROD,RLT_14,12067.0,12067.0
7,2024,10007,8002,FIN,50007,8002.0,PROD,RLT_16,12023.0,12023.0
8,2024,10008,8002,FIN,50008,8002.0,PROD,RLT_16,12067.0,12067.0
9,2024,10009,8002,FIN,50009,8002.0,PROD,RLT_14,12023.0,12023.0


Below, we can view all FIN materials and select the one to build the hierarchy for

In [34]:
fin_materials = data_grouped[data_grouped['produced_material_release_type'] == 'FIN'].copy()
print(fin_materials[["plant_id","year", "produced_material", "produced_material_release_type"]])

  plant_id  year  produced_material produced_material_release_type
0   RLT_10  2024              10000                            FIN
1   RLT_10  2024              10001                            FIN
2   RLT_14  2024              10002                            FIN
3   RLT_14  2024              10003                            FIN
4   RLT_14  2024              10004                            FIN
5   RLT_14  2024              10005                            FIN
6   RLT_14  2024              10006                            FIN
7   RLT_16  2024              10007                            FIN
8   RLT_16  2024              10008                            FIN
9   RLT_14  2024              10009                            FIN


In the next block I define a Level 1 material for hierarchy.
Also to optimize performance, I subset the initial dataset by year and plant.


In [35]:
#target_plant = 'RLT_10'
target_year = 2024
#target_material = 10000

data_grouped_subset = data_grouped[#(data_grouped["plant_id"]==target_plant) &
              (data_grouped["year"]==target_year) ].copy()

hierarchy= data_grouped_subset[#(data_grouped_subset["produced_material"]==target_material) &
              (data_grouped_subset["year"]==target_year) &
              #(data_grouped_subset["plant_id"]==target_plant) &
             (data_grouped_subset['produced_material_release_type'] == 'FIN')].copy()

hierarchy["level"] = 1
hierarchy['fin_material_id'] = hierarchy['produced_material']
hierarchy['fin_material_release_type'] = hierarchy['produced_material_release_type']
hierarchy['fin_material_production_type'] = hierarchy['produced_material_production_type']
hierarchy['fin_production_quantity'] = hierarchy['produced_material_quantity']


# data_grouped_subset.head()
hierarchy.head(15)


,year,produced_material,produced_material_production_type,produced_material_release_type,component_material,component_material_production_type,component_material_release_type,plant_id,produced_material_quantity,component_material_quantity,level,fin_material_id,fin_material_release_type,fin_material_production_type,fin_production_quantity
0,2024,10000,8002,FIN,50000,8002.0,PROD,RLT_10,11708.0,11708.0,1,10000,FIN,8002,11708.0
1,2024,10001,8002,FIN,50001,8002.0,PROD,RLT_10,12023.0,12023.0,1,10001,FIN,8002,12023.0
2,2024,10002,8002,FIN,50002,8002.0,PROD,RLT_14,12067.0,12067.0,1,10002,FIN,8002,12067.0
3,2024,10003,8002,FIN,50003,8002.0,PROD,RLT_14,12091.0,12091.0,1,10003,FIN,8002,12091.0
4,2024,10004,8002,FIN,50004,8002.0,PROD,RLT_14,12091.0,12091.0,1,10004,FIN,8002,12091.0
5,2024,10005,8002,FIN,50005,8002.0,PROD,RLT_14,12023.0,12023.0,1,10005,FIN,8002,12023.0
6,2024,10006,8002,FIN,50006,8002.0,PROD,RLT_14,12067.0,12067.0,1,10006,FIN,8002,12067.0
7,2024,10007,8002,FIN,50007,8002.0,PROD,RLT_16,12023.0,12023.0,1,10007,FIN,8002,12023.0
8,2024,10008,8002,FIN,50008,8002.0,PROD,RLT_16,12067.0,12067.0,1,10008,FIN,8002,12067.0
9,2024,10009,8002,FIN,50009,8002.0,PROD,RLT_14,12023.0,12023.0,1,10009,FIN,8002,12023.0


Create a loop and UNION ALL to build a hierarchy

In [36]:
current_level = hierarchy.copy()

while True:
  next_level = data_grouped_subset.merge(
            current_level[['component_material', 'plant_id', 'year', 'level','fin_material_id','fin_material_release_type','fin_material_production_type','fin_production_quantity']],
            left_on=['produced_material', 'plant_id', 'year'],
            right_on=['component_material', 'plant_id', 'year'],
            suffixes=('', '_parent')
        ).drop(columns=['component_material_parent'])

  if next_level.empty:
     break

  next_level['level'] +=1

  hierarchy = pd.concat([hierarchy, next_level], ignore_index=True)
  current_level = next_level

hierarchy.head(10)


,year,produced_material,produced_material_production_type,produced_material_release_type,component_material,component_material_production_type,component_material_release_type,plant_id,produced_material_quantity,component_material_quantity,level,fin_material_id,fin_material_release_type,fin_material_production_type,fin_production_quantity
0,2024,10000,8002,FIN,50000,8002.0,PROD,RLT_10,11708.0,11708.0,1,10000,FIN,8002,11708.0
1,2024,10001,8002,FIN,50001,8002.0,PROD,RLT_10,12023.0,12023.0,1,10001,FIN,8002,12023.0
2,2024,10002,8002,FIN,50002,8002.0,PROD,RLT_14,12067.0,12067.0,1,10002,FIN,8002,12067.0
3,2024,10003,8002,FIN,50003,8002.0,PROD,RLT_14,12091.0,12091.0,1,10003,FIN,8002,12091.0
4,2024,10004,8002,FIN,50004,8002.0,PROD,RLT_14,12091.0,12091.0,1,10004,FIN,8002,12091.0
5,2024,10005,8002,FIN,50005,8002.0,PROD,RLT_14,12023.0,12023.0,1,10005,FIN,8002,12023.0
6,2024,10006,8002,FIN,50006,8002.0,PROD,RLT_14,12067.0,12067.0,1,10006,FIN,8002,12067.0
7,2024,10007,8002,FIN,50007,8002.0,PROD,RLT_16,12023.0,12023.0,1,10007,FIN,8002,12023.0
8,2024,10008,8002,FIN,50008,8002.0,PROD,RLT_16,12067.0,12067.0,1,10008,FIN,8002,12067.0
9,2024,10009,8002,FIN,50009,8002.0,PROD,RLT_14,12023.0,12023.0,1,10009,FIN,8002,12023.0


rename columns

In [37]:
hierarchy = hierarchy.rename(columns={
    'plant_id':'plant',
    'produced_material':'prod_material_id',
    'produced_material_production_type':'prod_material_production_type',
    'produced_material_release_type':'prod_material_release_type',
    'produced_material_quantity':'prod_material_production_quantity',
    'component_material':'component_id',
    'component_material_production_type':'component_material_production_type',
    'component_material_release_type':'component_material_release_type',
    'component_material_quantity':'component_consumption_quantity'
})

#hierarchy.head(15)

FINAL!
hide the 1st row with fin_material, and arrange the columns in the requiered order

In [38]:
column_order = ['plant','fin_material_id','fin_material_release_type','fin_material_production_type','fin_production_quantity','prod_material_id','prod_material_release_type',
           'prod_material_production_type','prod_material_production_quantity', 'component_id', 'component_material_release_type', 'component_material_production_type', 'component_consumption_quantity', 'year', 'level']
hierarchy = hierarchy[column_order]


In [39]:
hres = hierarchy[hierarchy["level"] != 1]
hres.sort_values(by = ["fin_material_id", "level"]).head(12)

,plant,fin_material_id,fin_material_release_type,fin_material_production_type,fin_production_quantity,prod_material_id,prod_material_release_type,prod_material_production_type,prod_material_production_quantity,component_id,component_material_release_type,component_material_production_type,component_consumption_quantity,year,level
10,RLT_10,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,80070,PROD,8007.0,11303.0,2024,2
11,RLT_10,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,90000,ADD,NaN,598.0,2024,2
12,RLT_10,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,90001,ADD,NaN,242.0,2024,2
40,RLT_10,10000,FIN,8002,11708.0,80070,PROD,8007,11028.0,80010,PROD,8001.0,41769.0,2024,3
41,RLT_10,10000,FIN,8002,11708.0,80070,PROD,8007,11028.0,90002,ADD,NaN,355.0,2024,3
42,RLT_10,10000,FIN,8002,11708.0,80070,PROD,8007,11028.0,90003,ADD,NaN,121.0,2024,3
70,RLT_10,10000,FIN,8002,11708.0,80010,PROD,8001,21013.0,80000,PROD,8000.0,23360.0,2024,4
71,RLT_10,10000,FIN,8002,11708.0,80010,PROD,8001,21013.0,90004,ADD,NaN,1242.0,2024,4
90,RLT_10,10000,FIN,8002,11708.0,80000,PROD,8000,23478.0,70000,RM,NaN,30498.0,2024,5
91,RLT_10,10000,FIN,8002,11708.0,80000,PROD,8000,23478.0,90005,ADD,NaN,829.0,2024,5


In [40]:
hres[hres["fin_material_id"] == 10001].head(20)

,plant,fin_material_id,fin_material_release_type,fin_material_production_type,fin_production_quantity,prod_material_id,prod_material_release_type,prod_material_production_type,prod_material_production_quantity,component_id,component_material_release_type,component_material_production_type,component_consumption_quantity,year,level
13,RLT_10,10001,FIN,8002,12023.0,50001,PROD,8002,9487.0,80071,PROD,8007.0,10759.0,2024,2
14,RLT_10,10001,FIN,8002,12023.0,50001,PROD,8002,9487.0,90005,ADD,NaN,585.0,2024,2
15,RLT_10,10001,FIN,8002,12023.0,50001,PROD,8002,9487.0,90006,ADD,NaN,240.0,2024,2
43,RLT_10,10001,FIN,8002,12023.0,80071,PROD,8007,10751.0,80011,PROD,8001.0,42650.0,2024,3
44,RLT_10,10001,FIN,8002,12023.0,80071,PROD,8007,10751.0,90007,ADD,NaN,365.0,2024,3
45,RLT_10,10001,FIN,8002,12023.0,80071,PROD,8007,10751.0,90008,ADD,NaN,122.0,2024,3
72,RLT_10,10001,FIN,8002,12023.0,80011,PROD,8001,21300.0,80001,PROD,8000.0,24730.0,2024,4
73,RLT_10,10001,FIN,8002,12023.0,80011,PROD,8001,21300.0,90009,ADD,NaN,1214.0,2024,4
92,RLT_10,10001,FIN,8002,12023.0,80001,PROD,8000,24238.0,70001,RM,NaN,30624.0,2024,5
93,RLT_10,10001,FIN,8002,12023.0,80001,PROD,8000,24238.0,90010,ADD,NaN,830.0,2024,5
